# Pipeline Demo Notebook

This notebook illustrates the pipeline's functions in a minimal implementation.


## Step 1: Read Data & select markers

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import skimage
import pandas as pd
import anndata as ann

from scripts.normalize import _normalize_intensity
from scripts.max_projection import max_projection_hwc
from scripts.segment import segment_image
from scripts.extract_export import _compute_centroids_and_shapes, _compute_intensity_features, _label_ids


## Step 1: Configure paths and markers

Adjust these parameters to point to your data. The structure mirrors the
`config/config.yaml` defaults but is kept explicit here for clarity.


In [ ]:
data_p = Path("./example_data")
dapi_p = data_p / "DAPI.png"
marker_channels = ["cd3", "hladr", "cd20", "cd11c", "cd44", "vimentin", "plasmacell"]
norm_method = "minmax"  # 'percentile'
norm_params = {"percentiles": [1.0, 99.0]}

files = sorted(p for p in data_p.iterdir() if p.suffix.lower() == ".png"
 and p.is_file() and "dapi" not in str(p).lower())

model_name = "cellpose"
hyperparameters = {
    "cellpose": {
        "flow_threshold": 0.4,
        "cellprob_threshold": 0.8,
        "model_type": None,
        "model_path": None,
        "use_gpu": True,
    }
}

tiling = {
    "enabled": True,
    "tile_size": [256, 256],
    "overlap": 64,
}

min_cell_size = 250 


## Step 2: Read data & select markers

Use `scripts.read_data` to locate image files and build a stacked array with
DAPI + selected markers.


In [ ]:
# Find all non-DAPI .png files in the data root
stack = []

dapi_img = skimage.io.imread(dapi_p)
stack.append(dapi_img)
# Stack DAPI + selected marker channels into a single array (H, W, C)
for f in files:
    if not f.exists():
        raise FileNotFoundError(f"Input path not found: {f}")
    if any(marker in str(f).lower() for marker in marker_channels):
        img = skimage.io.imread(f)
        stack.append(img)

stack = np.stack(stack, axis=-1)
print(f"Stacked image shape: {stack.shape}")

#skimage.io.imsave("./results/out/stacked.tif", stack)

## Step 3: Normalize intensity

Delegate intensity normalization to `scripts.normalize.normalize_intensity`.


In [ ]:

stack_norm = _normalize_intensity(stack, method="percentile", params=norm_params)
print(stack_norm.shape, stack_norm.min(), stack_norm.max())

## Step 4: Max projection visualization image

Use `scripts.max_projection.make_max_projection` to build a 3-channel RGB-like
image for visualization: [DAPI, membrane projection, zeros].


In [ ]:
max_proj = max_projection_hwc(stack_norm)
plt.figure(figsize=(6, 6))
plt.imshow(max_proj)
plt.axis("off")
plt.title("Max projection")


## Step 5: Segment (optionally tiled)

In [ ]:
masks_untiled = segment_image(max_proj, model_name=model_name, hyper=hyperparameters.get(model_name, {}), tiling={})
masks_tiled = segment_image(max_proj, model_name=model_name, hyper=hyperparameters.get(model_name, {}), tiling=tiling)
rgb = np.zeros((masks_untiled.shape[0], masks_tiled.shape[1], 3), dtype=np.uint8)
masks_tiled = skimage.measure.label(masks_tiled) # Necessary after tiling for contiguous labels
rgb[masks_untiled > 0] = np.array([255, 0, 0], dtype=np.uint8)  # Untiled: Red
rgb[masks_tiled > 0] += np.array([0, 255, 0], dtype=np.uint8)  # Tiled: green

In [ ]:
# Expect minor differences in segmentation due to tiling
fig, axs = plt.subplots(1, 3, figsize=(12, 6))
axs[0].imshow(skimage.color.label2rgb(masks_untiled,))
axs[0].axis("off")
axs[0].set_title(f"Direct segmentation: {masks_untiled.max()} masks")
axs[1].imshow(skimage.color.label2rgb(masks_tiled,))
axs[1].axis("off")
axs[1].set_title(f"Tiled segmentation: {masks_tiled.max()} masks")
axs[2].imshow(rgb)
axs[2].axis("off")
axs[2].set_title(f"Differences in green/red");

## Step 6: Postprocess masks

Apply the same size-based filtering and relabelling as the Snakemake rule, via
`scripts.postprocess.postprocess_mask`.


In [ ]:
pp_masks = skimage.morphology.remove_small_objects(masks_tiled, max_size=min_cell_size)
out_mask = skimage.measure.label(pp_masks).astype(np.uint16)

plt.figure(figsize=(6, 6))
plt.imshow(skimage.color.label2rgb(pp_masks))
plt.axis("off")
plt.title("Postprocessed masks")


## Step 7: Extract per-cell features and build AnnData

Use `scripts.extract_export` utilities to compute regionprops, per-channel means,
        and construct an AnnData object. The details are delegated to the script so the
        notebook remains thin.


In [ ]:
label_ids = _label_ids(pp_masks)

y, x, shape_features, shape_names = _compute_centroids_and_shapes(pp_masks, label_ids, export_shapes=True)

intensity, intensity_names = _compute_intensity_features(pp_masks, label_ids, files)

feature_blocks = []
var_names = []

feature_blocks.append(intensity)
var_names.extend(intensity_names)

feature_blocks.append(shape_features)
var_names.extend(shape_names)

X = np.concatenate(feature_blocks, axis=1)

obs = pd.DataFrame(
    {"x": x, "y": y},
    index=pd.Index(label_ids.astype(str), name="Cell_ID"),
)
var = pd.DataFrame(index=pd.Index([str(name) for name in var_names], name="feature"))

adata = ann.AnnData(X=X, obs=obs, var=var)
adata.obsm["spatial"] = np.column_stack([x, y]).astype(np.float32, copy=False)
adata